In [ ]:
import numpy as np
import random
import math
import heapq

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

class HNSW:
    """A simplified Hierarchical Navigable Small World index built from scratch."""

    def __init__(self, dim, M=5, ef_construction=20, m_L=1.0):
        self.dim = dim
        self.M = M                        # max neighbors per node per layer
        self.ef_construction = ef_construction
        self.m_L = m_L                    # controls layer-assignment probability
        self.vectors = {}                 # id -> vector
        self.graph = {}                   # layer -> {id -> set(neighbor ids)}
        self.entry_point = None
        self.max_layer = -1

    def _random_layer(self):
        # probability-driven layer skipping: higher layers are exponentially rarer
        return int(-math.log(random.random()) * self.m_L)

    def _search_layer(self, query, entry_points, layer, ef):
        visited = set(entry_points)
        candidates = [(-cosine_sim(query, self.vectors[ep]), ep) for ep in entry_points]
        heapq.heapify(candidates)
        results = [(-c[0], c[1]) for c in candidates]

        while candidates:
            neg_sim, current = heapq.heappop(candidates)
            if -neg_sim < min(r[0] for r in results) and len(results) >= ef:
                break
            for neighbor in self.graph.get(layer, {}).get(current, []):
                if neighbor not in visited:
                    visited.add(neighbor)
                    sim = cosine_sim(query, self.vectors[neighbor])
                    heapq.heappush(candidates, (-sim, neighbor))
                    results.append((sim, neighbor))
                    results.sort(key=lambda x: -x[0])
                    results = results[:ef]
        return results

    def insert(self, node_id, vector):
        self.vectors[node_id] = np.array(vector)
        layer = self._random_layer()

        for l in range(layer + 1):
            self.graph.setdefault(l, {}).setdefault(node_id, set())

        if self.entry_point is None:
            self.entry_point = node_id
            self.max_layer = layer
            return

        ep = [self.entry_point]
        # traverse from top layer down to layer+1 just to find a good entry point
        for l in range(self.max_layer, layer, -1):
            ep = [r[1] for r in self._search_layer(vector, ep, l, ef=1)]

        # connect at each layer from min(layer, max_layer) down to 0
        for l in range(min(layer, self.max_layer), -1, -1):
            candidates = self._search_layer(vector, ep, l, self.ef_construction)
            neighbors = [c[1] for c in candidates[: self.M]]
            for n in neighbors:
                self.graph[l][node_id].add(n)
                self.graph[l].setdefault(n, set()).add(node_id)
            ep = [c[1] for c in candidates]

        if layer > self.max_layer:
            self.max_layer = layer
            self.entry_point = node_id

    def search(self, query, k=5, ef=20):
        query = np.array(query)
        ep = [self.entry_point]
        for l in range(self.max_layer, 0, -1):
            ep = [r[1] for r in self._search_layer(query, ep, l, ef=1)]
        results = self._search_layer(query, ep, 0, ef)
        results.sort(key=lambda x: -x[0])
        return results[:k]           # list of (cosine_similarity, id) -- O(log N) traversal


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    index = HNSW(dim=16)
    for i in range(200):
        index.insert(i, np.random.randn(16))

    query = np.random.randn(16)
    top5 = index.search(query, k=5)
    print("Top-5 nearest neighbors (similarity, id):")
    for sim, idx in top5:
        print(f"  {sim:.4f}  ->  node {idx}")

---
## Task 4: In-Memory HNSW Vector Indexing from Scratch